# Plot r_aven from data-collection
This notebook loads `*_filtered_dataset_correct_matrices.jsonl` files from `data-collection` for the `r_aven` variant, computes Raven (index 11) accuracy per hop, saves a CSV summary, and writes PNG+PDF plots.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
# Configuration
DATA_ROOT = Path("/home/abasso_aims_ac_za/divergence-tokens/data-collection/data/no-sys-prompt-original/r_aven/datasets")
OUTPUT_DIR = Path("/home/abasso_aims_ac_za/divergence-tokens/notebooks/eval-pref-with-stats/plots/r_aven")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAVEN_INDEX = 11  # 0-based index for raven row

In [ ]:
def read_matrices_from_jsonl(path):
    mats = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                # fall back: some files may be raw Python lists, try eval safely
                obj = eval(line)
            mats.append(obj)
    return mats

In [ ]:
rows = []
for p in sorted(DATA_ROOT.glob('hop*_filtered_dataset_correct_matrices.jsonl')):
    hop_name = p.stem.split('_')[0]  # e.g., hop0
    print('Processing', p)
    mats = read_matrices_from_jsonl(p)
    # For each sample (matrix), get raven row and compute fraction True across tokens
    sample_fracs = []
    for m in mats:
        try:
            raven_row = m[RAVEN_INDEX]
        except Exception:
            # if matrix shaped differently, skip sample
            continue
        arr = np.array(raven_row, dtype=bool)
        if arr.size == 0:
            continue
        sample_fracs.append(arr.mean())
    if len(sample_fracs) == 0:
        mean = float('nan')
        std = float('nan')
    else:
        mean = float(np.mean(sample_fracs))
        std = float(np.std(sample_fracs, ddof=1)) if len(sample_fracs) > 1 else 0.0
    rows.append({'hop': hop_name, 'n_samples': len(sample_fracs), 'raven_mean': mean, 'raven_std': std})
df = pd.DataFrame(rows)
df = df.sort_values('hop').reset_index(drop=True)
csv_out = OUTPUT_DIR / 'r_aven_raven_accuracy_by_hop.csv'
df.to_csv(csv_out, index=False)
print('Wrote', csv_out)

In [ ]:
# Plot mean with error bars (std) and save PNG+PDF
plt.figure(figsize=(8,4))
x = range(len(df))
plt.errorbar(x, df['raven_mean']*100, yerr=(df['raven_std']*100), fmt='-o')
plt.xticks(x, df['hop'], rotation=45)
plt.xlabel('Hop')
plt.ylabel('Raven accuracy (%)')
plt.title('Raven accuracy by hop (from data-collection)')
png = OUTPUT_DIR / 'r_aven_raven_accuracy_by_hop.png'
pdf = OUTPUT_DIR / 'r_aven_raven_accuracy_by_hop.pdf'
plt.tight_layout()
plt.savefig(png, dpi=300)
plt.savefig(pdf)
print('Saved', png, pdf)
plt.show()

## Notes
- This notebook reads the `*_filtered_dataset_correct_matrices.jsonl` files present in `data-collection`.
- It computes a sample-level raven-row token-accuracy (fraction True across token positions) and then reports mean/std across samples per hop.
- If you want per-token summaries, divergence analyses, or other metrics, I can extend this notebook with additional analyses (Jaccard, entropy, persistence, etc.).